# Architecture lead — GPU benchmark continuation

Use a Colab or Jupyter CUDA GPU runtime. Clone/open your team repository with your own GitHub access first and set its root as the working directory. Run all three assigned models on this common hardware/precision. The Windows float32 results remain separate exploratory evidence.

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path.cwd()
assert (ROOT/'a7/benchmark_runner.py').exists(), 'Open the team repository root first.'
os.environ['HF_HOME'] = str(ROOT/'model_cache/huggingface')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'a7/requirements.txt')])


Approve Llama access with your Hugging Face account, then use the login widget. Do not place a token in a code cell or share outputs containing credentials.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import json, platform, importlib.metadata
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before benchmarking.'
from a7.benchmark_runner import run_matrix, select_models
out = ROOT/'a7/results/architecture_benchmark.csv'
assert not out.exists(), 'Use a clean GPU checkout, or remove this guard only to resume the SAME GPU/dtype run.'
environment = dict(recorded_at_utc=datetime.now(timezone.utc).isoformat(),
    platform=platform.platform(), device='cuda', dtype='float16',
    gpu=torch.cuda.get_device_name(), vram_bytes=torch.cuda.get_device_properties(0).total_memory,
    max_new_tokens=220, do_sample=False,
    packages={p:importlib.metadata.version(p) for p in ['torch','transformers','huggingface-hub']})
out.parent.mkdir(parents=True,exist_ok=True)
out.with_name('architecture_benchmark_environment.json').write_text(json.dumps(environment,indent=2))
rows = run_matrix(select_models(owner='architecture'), out, device_request='cuda', dtype_request='float16')
print('Successful:',sum(r['status']=='ok' for r in rows),'/ 75')


In [ ]:
from a7.review_results import review_csv
review_csv(out)
print('Review every raw response and score before submission.')
print('Return architecture_benchmark.csv and architecture_benchmark_environment.json to your local checkout.')
print('Preserve CPU evidence separately before replacing the canonical input file.')
